# Navegar e verificar lineage

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="08_auditoria_lineage"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '1',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 3,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'snapshot',
 'external_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Escolha proposta, run, modelo+versão, snapshot ou campanha; paginação limita o grafo retornado.

In [2]:
from src.workbench.lineage_review import inspect_graph
external_id=params.external_id or (str(params.proposal_id) if params.proposal_id else "")
if not external_id: raise ValueError("Informe external_id ou proposal_id")
evidence=inspect_graph(cfg,params.lineage_kind,external_id,params.model_version)
display(pd.DataFrame(evidence["nodes"]));display(pd.DataFrame(evidence["edges"]))


,entity_id,entity_type,external_id,environment,domain,created_at_utc,attributes
0,57f60a7f-780d-45b6-bb3a-b24bcc5ba7fc,campaign,1e148711-47b3-4750-9346-f4df35def14e,poc,planilha-turmas,2026-11-04T21:43:27.992645+00:00,{}
1,e76ec16b-b0b4-4c53-997a-7f9eb8a8ece3,campaign,272115ed-4d91-48ba-b863-5aabd49dbf0a,poc,planilha-turmas,2026-11-04T21:41:48.439642+00:00,{}
2,5d3d6370-1fbf-405d-89b0-550a668285ef,campaign,2a9e2e23-0173-4d19-83f9-129ce49affff,poc,planilha-turmas,2026-11-04T21:26:04.718503+00:00,{}
3,2a99e02e-ae74-4e88-825f-ebaaffd39b40,campaign,2ecb5a7f-4d8d-48e9-bfc4-cbafc347c736,poc,planilha-turmas,2026-11-04T21:41:39.463748+00:00,{}
4,c509d9cf-ffc9-428a-9a7d-af7dbf93cbbc,campaign,4c559b77-6699-4fe2-9ade-dce8ecd33c90,poc,planilha-turmas,2026-11-04T21:43:07.053302+00:00,{}
...,...,...,...,...,...,...,...
95,60b6ba0c-86a6-4b06-bd38-81691e2cff21,run,a80447d4938943a3bd2729fa0dc13ca6,poc,planilha-turmas,2026-11-04T21:43:46.987846+00:00,{}
96,669873b6-e35a-472f-a611-7945fbd1770b,run,a80e9b36e4764129b935ff4ff804fc71,poc,planilha-turmas,2026-11-04T21:43:29.601718+00:00,{}
97,a40534c5-0028-4ea2-9317-0aca715d2bcb,run,b5abd260fc654932aeb765ed2afd8f8f,poc,planilha-turmas,2026-11-04T21:43:36.179643+00:00,{}
98,6288bada-6737-4efa-a84a-ff3e490c8fc7,run,b71ccc3e15e844bd9b52c3324a768fb6,poc,planilha-turmas,2026-11-04T21:43:39.138594+00:00,{}


,edge_id,parent_id,child_id,relation_type,created_at_utc
0,02fbe40f-3c9d-4036-b7ca-eb358d7f2bc4,7a4954c5-2f7f-49c7-9ba6-8a7f7c0cc8ba,1bc84259-d8ad-4f41-a37c-b17aa429e56c,REGISTERED_AS,2026-11-04T21:43:39.132552+00:00
1,0604287a-b888-43b2-83c1-077e986f474b,05eb9f18-9fc2-448d-998f-19111d7b712f,71f38808-5da5-4291-b992-f471b4bb83a2,REGISTERED_AS,2026-11-04T21:43:31.042679+00:00
2,07158669-4450-44cd-b95b-f33e247dc2bf,a5ba057e-fbab-4b31-925e-d036e7c5219b,8923a0ea-d6c0-472a-9f71-a6667ede2059,REGISTERED_AS,2026-11-04T21:43:32.437473+00:00
3,097db15f-5ed5-4749-bf14-37e5356a9e48,57f60a7f-780d-45b6-bb3a-b24bcc5ba7fc,c400015d-ec7e-49f8-8374-f1066a1d6844,CONTRIBUTED_TO,2026-11-04T21:43:34.940243+00:00
4,09a7924f-c531-4080-930b-bfadd31e9034,bcf7c32e-4ad8-495a-983b-d1315b7e7b43,a487aa73-b1af-4df2-bf14-c0526c8a42fd,REGISTERED_AS,2026-11-04T21:43:33.685273+00:00
...,...,...,...,...,...
95,a5446d95-ded5-4f3b-86aa-4bfb4bcab6be,e76ec16b-b0b4-4c53-997a-7f9eb8a8ece3,35c66c5d-d582-4d61-9075-eb39e254974e,CONTRIBUTED_TO,2026-11-04T21:41:48.439642+00:00
96,aa5f01d7-2f5d-447c-9cd3-6f28b27b0163,57f60a7f-780d-45b6-bb3a-b24bcc5ba7fc,8c1dfaac-8490-45fd-964e-8ac597f98f01,CONTRIBUTED_TO,2026-11-04T21:43:27.997028+00:00
97,ab98164d-bd2d-47fb-932e-df2768bd9d3f,a8de65d8-97df-4d09-ba0d-8cb2d9a676e3,76cfbd6a-6f58-4574-847a-d16f33cdda28,PREDICTED_WITH,2026-11-04T21:41:41.732445+00:00
98,ac6bcbd8-a7a7-4904-8f81-527c3496e425,57f60a7f-780d-45b6-bb3a-b24bcc5ba7fc,05eb9f18-9fc2-448d-998f-19111d7b712f,CONTRIBUTED_TO,2026-11-04T21:43:31.042679+00:00


2. Grafo e verificações reais. VERIFIED exige presença/proveniência; MISSING/MISMATCH bloqueiam confiança; legado continua identificado.

In [3]:
from IPython.display import HTML
from src.workbench.lineage_review import render_graph
display(HTML(render_graph(evidence)))
display(pd.DataFrame(evidence["checks"]))
reconciled=audit(cfg,params.lineage_kind,external_id,params.model_version)
display(reconciled)
inventory_rows=call(cfg,"GET","/v1/audit/inventory?limit=500")
display(pd.DataFrame(inventory_rows))
display(export_session(reconciled,inventory_rows))


run: 6a45925fe11e4172bd8497db80286b97,→ REGISTERED_AS →,model: hi.pt.demanda.DEMO-C1.DEMO-CR5.P2/1
run: 0758ffa587b34d1fba74459d29fed5b7,→ REGISTERED_AS →,model: hi.pt.demanda.DEMO-C3.DEMO-CR4.P1/1
run: 41c374d45dfe4da1bbe9975cf8f8aac1,→ REGISTERED_AS →,model: hi.pt.demanda.DEMO-C2.DEMO-CR2.P2/1
campaign: 1e148711-47b3-4750-9346-f4df35def14e,→ CONTRIBUTED_TO →,c400015d-ec7e-49f8-8374-f1066a1d6844
run: 08f3cd64f1d74d9590200cf3ee5389f2,→ REGISTERED_AS →,model: hi.pt.demanda.DEMO-C2.DEMO-CR3.P1/1
model: hi.demo.notebook.a9291c01/1,→ PREDICTED_WITH →,inference: 2af2e452-b78c-4a86-a735-92f3276f9baa
inference: 3fd2478b-0ae7-4cb0-853f-c5e3aa9580aa,→ CONTRIBUTED_TO →,proposal: 37ce4733-c129-4404-997d-a9f630d81da9
522cf1dc-5ab7-4e49-af1b-aaf393570f95,→ TRAINED_ON →,run: 1932809c8fa2454a9d6a86920ea9e707
campaign: 1e148711-47b3-4750-9346-f4df35def14e,→ CONTRIBUTED_TO →,run: 2e3be451004e4dfe91ccca53258d8fdb
dc9ddb22-2fc2-4edb-8b70-e68ac5f3dcec,→ REGISTERED_AS →,model: hi.pt.demanda.DEMO-C1.DEMO-CR2.P1/1
run: 420b1e103c8741ddbb8ec66427a95c0b,→ REGISTERED_AS →,model: hi.demo.notebook.a9291c01/1


,kind,id,status,evidence,url
0,campaign,1e148711-47b3-4750-9346-f4df35def14e,VERIFIED,PostgreSQL lineage entity,NaN
1,campaign,272115ed-4d91-48ba-b863-5aabd49dbf0a,VERIFIED,PostgreSQL lineage entity,NaN
2,campaign,2a9e2e23-0173-4d19-83f9-129ce49affff,VERIFIED,PostgreSQL lineage entity,NaN
3,campaign,2ecb5a7f-4d8d-48e9-bfc4-cbafc347c736,VERIFIED,PostgreSQL lineage entity,NaN
4,campaign,4c559b77-6699-4fe2-9ade-dce8ecd33c90,VERIFIED,PostgreSQL lineage entity,NaN
...,...,...,...,...,...
95,run,a80447d4938943a3bd2729fa0dc13ca6,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/3/runs/a80...
96,run,a80e9b36e4764129b935ff4ff804fc71,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/3/runs/a80...
97,run,b5abd260fc654932aeb765ed2afd8f8f,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/3/runs/b5a...
98,run,b71ccc3e15e844bd9b52c3324a768fb6,VERIFIED,PostgreSQL lineage entity,http://127.0.0.1:5000/#/experiments/3/runs/b71...


{'inventory': [{'model_key': 'hi.demo.integration.70491a51',
   'model_version': '1',
   'run_id': '466af8046be248029c27e23cce2ee70e',
   'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
   'campaign_id': '272115ed-4d91-48ba-b863-5aabd49dbf0a',
   'model_display_name': 'Ensaio DEMO agregado • hi.demo.integration.70491a51',
   'cycle_code': '2026-1',
   'campus_code': None,
   'course_code': None,
   'period_code': None,
   'experiment_name': 'HI/POC/PlanilhaTurmas/Demanda/2026-1',
   'experiment_id': '3',
   'run_name': 'hi.demo.integration.70491a51__ridge__20261104T214147Z__a01__cd8f5a50',
   'artifact_uri': 'models:/m-2c18bfddf5ab4d07ad941200beecd0b3',
   'artifact_sha256': '4268666504b258b661b9c6cc009bdf8c21d06fde4317fb389a585e633be3ad0b',
   'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
   'algorithm': 'ridge',
   'manifest_uri': 's3://pucpr-poc/lineage/poc/planilha-turmas/catalog/cycles/2026-1/models/hi.demo.integration.70491a51/versio

,model_key,model_version,run_id,snapshot_id,campaign_id,model_display_name,cycle_code,campus_code,course_code,period_code,...,run_name,artifact_uri,artifact_sha256,dataset_sha256,algorithm,manifest_uri,manifest_sha256,integrity_status,sql_approved,metadata
0,hi.demo.integration.70491a51,1,466af8046be248029c27e23cce2ee70e,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,272115ed-4d91-48ba-b863-5aabd49dbf0a,Ensaio DEMO agregado • hi.demo.integration.704...,2026-1,None,None,None,...,hi.demo.integration.70491a51__ridge__20261104T...,models:/m-2c18bfddf5ab4d07ad941200beecd0b3,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,904c250f95d849a020f4d16179cf39da4bc3bc408ececb...,RECORDED,False,{'files': [{'key': 'mlflow/3/models/m-2c18bfdd...
1,hi.demo.notebook.a9291c01,1,420b1e103c8741ddbb8ec66427a95c0b,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,4c559b77-6699-4fe2-9ade-dce8ecd33c90,Ensaio DEMO agregado • hi.demo.notebook.a9291c01,2026-1,None,None,None,...,hi.demo.notebook.a9291c01__ridge__20261104T214...,models:/m-053c00350db44eb1a28c5dae0ed54a66,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,24c12ccb21e8a1fd003a5abba87736174cd287f04180d0...,RECORDED,True,{'files': [{'key': 'mlflow/3/models/m-053c0035...
2,hi.demo.probe.18cb82de,1,d6023330e3944990ac27ecf9fee47e51,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,2a9e2e23-0173-4d19-83f9-129ce49affff,Ensaio DEMO agregado • hi.demo.probe.18cb82de,2026-1,None,None,None,...,hi.demo.probe.18cb82de__ridge__20261104T212603...,models:/m-b5385d2b647d422189e0148b354a04f4,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,a1bdce8bf152b287529453a87475e26e9172ea10f12071...,RECORDED,False,{'files': [{'key': 'mlflow/3/models/m-b5385d2b...
3,hi.demo.v3-negative.699bb4f2,1,8de6f68bbf5c41b3a67a461e3d67775a,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,6fa95eaf-75a5-423d-b6bd-cf19612fb001,Ensaio DEMO agregado • hi.demo.v3-negative.699...,2026-1,None,None,None,...,hi.demo.v3-negative.699bb4f2__ridge__20261104T...,models:/m-fed11603f82a4ff6a0f3dd15aa85a7a5,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,140b692874660c83bb4f67346032a249be246c381fbe46...,RECORDED,False,{'files': [{'key': 'mlflow/3/models/m-fed11603...
4,hi.demo.v3-negative.f3c26d13,1,fd2908848d2545928a73061cfc8325a4,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,aac36ae8-f548-4ec8-ba4c-5bf2d5aca4e8,Ensaio DEMO agregado • hi.demo.v3-negative.f3c...,2026-1,None,None,None,...,hi.demo.v3-negative.f3c26d13__ridge__20261104T...,models:/m-e23fd62eadf2420985fdf46b9d6e84a2,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,f1669598a3b4601b9f2b992b884beb9f9cc0da0431392e...,RECORDED,False,{'files': [{'key': 'mlflow/3/models/m-e23fd62e...
5,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,3,19495cf828a946d2924858e554ce5641,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,1e148711-47b3-4750-9346-f4df35def14e,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...,2026-1,DEMO-C1,DEMO-CR1,P1,...,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1__ridge__2026...,models:/m-66f067f12cd1497dba47362003d0d78a,ed906b992cc3d85b0641488fa01758037fd33efd7bd6f3...,22b3c1491ef5f01f1c646bc43fd8350d324bce87b3bf4e...,ridge,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...,dee31bb7ccd6d5a26e84b9f73418742c4efde09dccd5a0...,RECORDED,False,{'files': [{'key': 'mlflow/3/models/m-66f067f1...
6,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,2,f6f552029db24af6bd4a1385b726017d,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,2ecb5a7f-4d8d-48e9-bfc4-cbafc347c736,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...,2026-1,DEMO-C1,DEMO-CR1,P1,...,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1__ridge__

'/reports/v3/manual-session-e69c178a-fd6d-4b59-af5f-7aa5629a82f2.json'

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [4]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v3/08_auditoria_lineage.json'